In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from scipy.stats import theilslopes
import pymannkendall as mk

In [ ]:
# --- LOAD + BASIC SETUP ---
df =      # Daily bare soil prediction timeseries over 10,000 points
era5_df = # ERA5 daily data extracted over the 10,000 points
INFOGPS = # Latitude and Longitude information for the 10,000 points

df['date'] = pd.to_datetime(df['date'])

era5_df['date'] = pd.to_datetime(era5_df['timeFormat'], errors='coerce')

# --- 0. APPLY SNOW FILTER (Daily Level) ---
# Merge daily flags with ERA5 data before aggregation
df = df.merge(era5_df[['POINTID', 'date', 'snow_d', 'airtemp_2m']], on=['POINTID', 'date'], how='left')

# Logic: If snow > 1cm --> Day is NOT BARE!
is_snowy = (df['snow_d'] > 0.01)
df['bare_soil_flag'] = np.where(is_snowy, 0, df['bare_soil_flag'])


# --- 1. DAILY → MONTHLY AGGREGATION -----

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

monthly_df = (
    df.groupby(['POINTID', 'year', 'month'])
    .agg(
        bare_soil_days=('bare_soil_flag', 'sum'),
        observation_days=('observation_flag', 'sum')
    )
    .reset_index()
)

# Add month metadata
monthly_df['date'] = pd.to_datetime(dict(year=monthly_df['year'], month=monthly_df['month'], day=1))
monthly_df['days_in_month'] = monthly_df['date'].dt.days_in_month


# --- 2. STRICT  QUALITY FILTER ---
start_date = pd.Timestamp('2017-12-01')
end_date   = pd.Timestamp('2024-12-01')

# Remove any points with incomplete timeseries 
is_in_window = (monthly_df['date'] >= start_date) & (monthly_df['date'] <= end_date)
is_dirty = monthly_df['observation_days'] < monthly_df['days_in_month']
dropped_ids = monthly_df[is_in_window & is_dirty]['POINTID'].unique()

df_clean = monthly_df[~monthly_df['POINTID'].isin(dropped_ids)].copy()


# --- 3. MAPPING THE REMOVALS ---
GPS_clean_map = INFOGPS[INFOGPS['POINTID'].isin(df_clean['POINTID'].unique())].drop_duplicates('POINTID')
GPS_dropped_map = INFOGPS[INFOGPS['POINTID'].isin(dropped_ids)].drop_duplicates('POINTID')

print(f"Points kept (96/96 months): {len(GPS_clean_map)}")
print(f"Points removed: {len(GPS_dropped_map)}")

projection = ccrs.LambertAzimuthalEqualArea(central_longitude=GPS_clean_map.longitude.mean(), 
                                            central_latitude=GPS_clean_map.latitude.mean())

fig, ax = plt.subplots(figsize=(7, 6), subplot_kw={'projection': projection})
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linestyle=':', alpha=0.5)

ax.scatter(GPS_clean_map.longitude, GPS_clean_map.latitude, s=3, color='tab:blue', 
           label="Cleaned (Continuous)", transform=ccrs.PlateCarree())
ax.scatter(GPS_dropped_map.longitude, GPS_dropped_map.latitude, s=3, color='darkorange', 
           label=f"Dropped (n={len(GPS_dropped_map)})", transform=ccrs.PlateCarree())

plt.legend(loc='upper left')
plt.title("Spatial Distribution of Data Quality Filtering")
plt.show()


# --- 4. ANNUAL (CALENDAR YEAR) TOTALS ---
calendar_start = pd.Timestamp('2018-01-01')
calendar_end   = pd.Timestamp('2024-12-31')

df_calendar = df_clean[
    (df_clean['date'] >= calendar_start) &
    (df_clean['date'] <= calendar_end)
].copy()

# Keep only complete years (12 months)
year_counts = df_calendar.groupby(['POINTID', 'year'])['month'].count().reset_index()
complete_years = year_counts[year_counts['month'] == 12]

annual_calendar_totals = (
    df_calendar.merge(complete_years[['POINTID', 'year']], on=['POINTID', 'year'])
    .groupby(['POINTID', 'year'])['bare_soil_days']
    .sum()
    .reset_index()
    .rename(columns={'bare_soil_days': 'total_annual_days'})
)

# --- 5. SEASONAL DATA (METEOROLOGICAL SEASONS) ---
seasonal_start = pd.Timestamp('2017-12-01')
seasonal_end   = pd.Timestamp('2024-11-30')

df_seasonal = df_clean[
    (df_clean['date'] >= seasonal_start) &
    (df_clean['date'] <= seasonal_end)
].copy()

# Assign seasonal year (Dec → next year)
df_seasonal['seasonal_year'] = df_seasonal['year']
df_seasonal.loc[df_seasonal['month'] == 12, 'seasonal_year'] += 1

# Season function
def get_season(month):
    if month in [3, 4, 5]: return "Spring"
    elif month in [6, 7, 8]: return "Summer"
    elif month in [9, 10, 11]: return "Autumn"
    else: return "Winter"

df_seasonal['season'] = df_seasonal['month'].apply(get_season)

# Aggregate seasonal totals
seasonal_data = (
    df_seasonal.groupby(['POINTID', 'seasonal_year', 'season'])
    .agg(
        seasonal_total_days=('bare_soil_days', 'sum'),
        n_months=('month', 'count')
    )
    .reset_index()
    .rename(columns={'seasonal_year': 'year'})
)

# Keep only full seasons (3 months)
seasonal_data = seasonal_data[seasonal_data['n_months'] == 3]

# --- 6. FINAL SUMMARIES (READY FOR PLOTTING) ---
# Annual summary
yearly_summary = (
    annual_calendar_totals
    .groupby('year')['total_annual_days']
    .agg(['mean', 'std'])
    .reset_index()
)

# Seasonal summary
seasonal_stats = (
    seasonal_data
    .groupby(['year', 'season'])['seasonal_total_days']
    .agg(['mean', 'std'])
    .reset_index()
)


plt.figure(figsize=(7, 5))
seasons = ["Spring", "Summer", "Autumn", "Winter"]
colors = sns.color_palette("husl", 4)

for i, season in enumerate(seasons):
    subset = seasonal_stats[seasonal_stats['season'] == season].sort_values('year')
    if subset.empty: continue
    
    x, y = subset['year'].values, subset['mean'].values
    res = theilslopes(y, x)
    mk_res = mk.original_test(y)
    
    sig_star = "*" if mk_res.p < 0.05 else ""

    #label = f"{reg}: {res[0]:.2f} days/yr{sig_level} (p={mk_test.p:.3f})"
    #frameon=False,
    plt.plot(x, y, marker='o', label=f"{season}{sig_star} {res[0]:.2f} days/yr (p: {mk_res.p:.3f})", color=colors[i], linewidth=2)
    plt.fill_between(x, np.maximum(0, y - subset['std']), y + subset['std'], color=colors[i], alpha=0.08)
    
    if mk_res.p < 0.05:
        plt.plot(x, res[1] + res[0] * x, linestyle='--', color="k", alpha=0.6)

plt.title("EU Seasonal Bare Soil Trends (Dec 2017 - Nov 2024)", fontsize=12)
plt.ylabel("Total Bare Soil Days per Season")
plt.xlabel("Seasonal Year (Winter includes Dec of previous year)")
plt.legend(frameon=False,loc='upper right')
plt.grid(True, linestyle=':', alpha=0.4)
plt.tight_layout()
plt.show()

# --- 7. PLOT ANNUAL CALENDAR TREND and CALCULATE STATS using Mann-Kendall test and Theil Sen slope ---
yearly_summary = annual_calendar_totals.groupby('year')['total_annual_days'].agg(['mean', 'std']).reset_index()
x_yr, y_yr = yearly_summary['year'].values, yearly_summary['mean'].values
res_yr = theilslopes(y_yr, x_yr)
mk_yr = mk.original_test(y_yr)

plt.figure(figsize=(6, 5))
plt.plot(x_yr, y_yr, marker='s', color='tab:blue', label="EU Annual Mean", linewidth=2.5)
plt.fill_between(x_yr, np.maximum(0, y_yr - yearly_summary['std']), y_yr + yearly_summary['std'], color='tab:blue', alpha=0.1)
plt.plot(x_yr, res_yr[1] + res_yr[0] * x_yr, linestyle='--', color='k', linewidth=2, label=f"Theil-Sen: {res_yr[0]:.2f} days/yr (p={mk_yr.p:.3f})")

plt.title("EU Total Annual Bare Soil Trend (Jan 2018 - Dec 2024)", fontweight='bold')
plt.ylabel("Total Bare Soil Days")
plt.xlabel("Year")
plt.ylim(bottom=0)
plt.grid(True, linestyle=':', alpha=0.4)
plt.legend(frameon=False,loc='upper right')
plt.tight_layout()

plt.show()

print(f"Annual Trend Result: {mk_yr.trend} (p-value: {mk_yr.p:.4f})")


